# ELSER Semantic Search - Interactive Workshop

**Time**: 30 minutes

In this notebook, you'll:
- Create ELSER-optimized indexes
- Load travel data with embeddings
- Run semantic search queries
- Test cross-lingual search
- Compare ELSER vs traditional search

---

## Setup

In [ ]:
import os
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import json

# Connect to Elastic (credentials from previous notebook)
es = Elasticsearch(
    cloud_id=os.getenv('ELASTIC_CLOUD_ID'),
    basic_auth=(os.getenv('ELASTIC_USERNAME'), os.getenv('ELASTIC_PASSWORD'))
)

print("✅ Connected to Elastic Cloud")

## Step 1: Create ELSER-Optimized Index

Let's create an index for travel cities with ELSER embeddings:

In [ ]:
INDEX_NAME = "travel-cities"
PIPELINE_NAME = f"{INDEX_NAME}-elser-pipeline"

# Delete if exists (for clean start)
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f"🗑️  Deleted existing index: {INDEX_NAME}")

# Create index mapping
mappings = {
    "properties": {
        "name": {"type": "keyword"},
        "country": {"type": "keyword"},
        "description": {"type": "text"},
        "description_embedding": {
            "type": "sparse_vector"  # ELSER embeddings
        },
        "highlights": {"type": "text"},
        "budget_level": {"type": "keyword"},
        "tags": {"type": "keyword"}
    }
}

es.indices.create(index=INDEX_NAME, mappings=mappings)
print(f"✅ Created index: {INDEX_NAME}")

# Create ELSER inference pipeline
es.ingest.put_pipeline(
    id=PIPELINE_NAME,
    description="ELSER pipeline for travel cities",
    processors=[
        {
            "inference": {
                "model_id": ".elser_model_2",
                "input_output": {
                    "input_field": "description",
                    "output_field": "description_embedding"
                }
            }
        }
    ]
)
print(f"✅ Created pipeline: {PIPELINE_NAME}")

## Step 2: Load Sample Data

Index some travel destinations with automatic ELSER embedding generation:

In [ ]:
# Sample travel data
cities = [
    {
        "name": "Tokyo",
        "country": "Japan",
        "description": "Tokyo is a vibrant metropolis that seamlessly blends ultra-modern technology with traditional Japanese culture. From neon-lit streets of Shibuya and serene temples of Asakusa to world-class dining and cutting-edge electronics, Tokyo offers an unforgettable experience for technology enthusiasts and culture lovers alike.",
        "highlights": ["Shibuya Crossing", "Senso-ji Temple", "Tokyo Skytree", "Tsukiji Market"],
        "budget_level": "medium",
        "tags": ["technology", "food", "culture", "shopping", "modern"]
    },
    {
        "name": "Paris",
        "country": "France",
        "description": "Paris, the City of Light, is renowned for its art, fashion, gastronomy, and culture. Home to iconic landmarks like the Eiffel Tower and Louvre Museum, Paris exudes romance with its charming cafes, elegant boulevards, and world-famous cuisine. Perfect for art lovers, history buffs, and anyone seeking a romantic getaway.",
        "highlights": ["Eiffel Tower", "Louvre Museum", "Notre-Dame", "Champs-Élysées"],
        "budget_level": "high",
        "tags": ["art", "culture", "romance", "food", "history"]
    },
    {
        "name": "Bali",
        "country": "Indonesia",
        "description": "Bali is a tropical paradise known for its stunning beaches, lush rice terraces, ancient temples, and vibrant culture. Whether you're seeking relaxation at luxury beach resorts, adventure through jungle treks, or spiritual experiences at sacred temples, Bali offers the perfect blend of natural beauty and cultural richness.",
        "highlights": ["Ubud Rice Terraces", "Tanah Lot Temple", "Seminyak Beach", "Sacred Monkey Forest"],
        "budget_level": "low",
        "tags": ["beach", "nature", "relaxation", "temples", "adventure"]
    },
    {
        "name": "New York",
        "country": "USA",
        "description": "New York City is the city that never sleeps, offering unparalleled urban energy, world-class museums, Broadway shows, and diverse culinary experiences. From the iconic Statue of Liberty and Times Square to Central Park and trendy Brooklyn neighborhoods, NYC provides endless excitement and cultural experiences.",
        "highlights": ["Statue of Liberty", "Times Square", "Central Park", "Broadway"],
        "budget_level": "high",
        "tags": ["urban", "culture", "entertainment", "food", "shopping"]
    },
    {
        "name": "Barcelona",
        "country": "Spain",
        "description": "Barcelona combines stunning architecture by Gaudí, beautiful Mediterranean beaches, vibrant nightlife, and rich Catalan culture. This coastal city offers perfect weather, world-renowned art museums, delicious tapas, and a relaxed beach lifestyle that attracts visitors year-round.",
        "highlights": ["Sagrada Família", "Park Güell", "La Rambla", "Gothic Quarter"],
        "budget_level": "medium",
        "tags": ["architecture", "beach", "culture", "nightlife", "food"]
    }
]

# Index documents with ELSER pipeline
actions = []
for city in cities:
    actions.append({
        "_index": INDEX_NAME,
        "_source": city,
        "pipeline": PIPELINE_NAME
    })

success, failed = bulk(es, actions, raise_on_error=False)

print(f"✅ Indexed {success} cities with ELSER embeddings")

# Refresh index
es.indices.refresh(index=INDEX_NAME)
print("✅ Index refreshed and ready for search!")

## Step 3: Semantic Search with ELSER

Now let's search using natural language queries!

### Example 1: "romantic destination with great food"

In [ ]:
def search_with_elser(query_text, top_k=3):
    """Search using ELSER semantic understanding"""
    
    search_body = {
        "query": {
            "text_expansion": {
                "description_embedding": {
                    "model_id": ".elser_model_2",
                    "model_text": query_text
                }
            }
        },
        "size": top_k,
        "_source": ["name", "country", "description", "tags", "budget_level"]
    }
    
    results = es.search(index=INDEX_NAME, body=search_body)
    
    print(f"\n🔍 Query: '{query_text}'")
    print(f"Found {results['hits']['total']['value']} results:\n")
    print("="*80)
    
    for i, hit in enumerate(results['hits']['hits'], 1):
        score = hit['_score']
        source = hit['_source']
        
        print(f"\n{i}. {source['name']}, {source['country']}")
        print(f"   Relevance Score: {score:.2f}")
        print(f"   Budget: {source['budget_level']}")
        print(f"   Tags: {', '.join(source['tags'][:4])}")
        print(f"   {source['description'][:150]}...")
    
    print("\n" + "="*80)
    return results

# Try it!
search_with_elser("romantic destination with great food")

### Example 2: "tech-savvy city with modern vibe"

Notice how ELSER understands concepts like "tech-savvy" and "modern"!

In [ ]:
search_with_elser("tech-savvy city with modern vibe")

### Example 3: "tropical paradise for relaxation"

In [ ]:
search_with_elser("tropical paradise for relaxation")

## Step 4: Cross-Lingual Search

**ELSER magic**: Search in one language, find results in any language!

### Search in Spanish:

In [ ]:
# Spanish query
search_with_elser("ciudad romántica con buena comida")  # "romantic city with good food"

### Search in French:

In [ ]:
# French query
search_with_elser("ville moderne avec technologie")  # "modern city with technology"

### Search in Japanese:

In [ ]:
# Japanese query
search_with_elser("ビーチリゾート")  # "beach resort"

## Step 5: Compare ELSER vs Traditional Search

Let's see the difference between ELSER semantic search and traditional keyword (BM25) search:

In [ ]:
def compare_search_methods(query_text):
    """Compare ELSER vs BM25 keyword search"""
    
    print(f"\n📊 COMPARING: '{query_text}'")
    print("="*80)
    
    # Traditional BM25 search
    print("\n1️⃣  TRADITIONAL KEYWORD SEARCH (BM25)")
    print("-"*80)
    
    bm25_body = {
        "query": {
            "multi_match": {
                "query": query_text,
                "fields": ["name", "description", "tags"]
            }
        },
        "size": 3
    }
    
    bm25_results = es.search(index=INDEX_NAME, body=bm25_body)
    
    if bm25_results['hits']['total']['value'] > 0:
        for i, hit in enumerate(bm25_results['hits']['hits'], 1):
            print(f"{i}. {hit['_source']['name']} (score: {hit['_score']:.2f})")
    else:
        print("❌ No results found!")
    
    # ELSER semantic search
    print("\n2️⃣  ELSER SEMANTIC SEARCH")
    print("-"*80)
    
    elser_body = {
        "query": {
            "text_expansion": {
                "description_embedding": {
                    "model_id": ".elser_model_2",
                    "model_text": query_text
                }
            }
        },
        "size": 3
    }
    
    elser_results = es.search(index=INDEX_NAME, body=elser_body)
    
    for i, hit in enumerate(elser_results['hits']['hits'], 1):
        print(f"{i}. {hit['_source']['name']} (score: {hit['_score']:.2f})")
    
    print("\n" + "="*80)
    print("💡 Notice: ELSER understands meaning, not just keywords!")

# Test with queries that require understanding
compare_search_methods("affordable beach vacation")

In [ ]:
# Another comparison
compare_search_methods("family-friendly adventure")

## Step 6: Try Your Own Queries!

Now it's your turn! Try different queries:

In [ ]:
# Try your own query here!
my_query = "city with amazing architecture and nightlife"  # ← Change this!

search_with_elser(my_query)

### More query ideas to try:

```python
"hidden gem for foodies"
"bustling urban center"
"spiritual and peaceful place"
"instagram-worthy destination"
"best for solo travelers"
"luxury honeymoon spot"
```

## 🎯 Key Takeaways

### What You Learned:

1. **Zero-shot Learning**: ELSER works without training data
2. **Semantic Understanding**: Understands meaning, not just keywords
3. **Cross-lingual**: Search in any language, find results in any language
4. **Context Aware**: Understands concepts like "romantic", "tech-savvy", "affordable"
5. **Better than Keywords**: Finds relevant results that keyword search misses

### ELSER Benefits:

✅ **No Training Required** - Works immediately  
✅ **Multi-lingual** - 100+ languages supported  
✅ **Runs in Elastic** - No external API calls  
✅ **Data Privacy** - All data stays in your AWS/Elastic  
✅ **Cost Effective** - Included in Elastic subscription  

---

## Next Steps:

Continue to **Notebook 02** - Building MCP Tools and Agents

Or explore more:
- Add more cities to the index
- Try filtering by budget_level
- Combine ELSER with traditional filters
- Index hotels and activities